<a href="https://colab.research.google.com/github/MaryamAmjad2/MultiCity-Weather-Forecast-LSTM/blob/main/MultiCity_Temp_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import matplotlib.pyplot as plt

# Device Agnostics

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load Data

In [ ]:
df_raw = pd.read_csv('pakistan_weather_2000_2024.csv')
df = df_raw.copy()

# PreProcessing

In [ ]:
# Drop 'visibility' column if it exists
if 'visibility' in df.columns:
    df = df.drop(columns=['visibility'])

# Handle missing values
df = df.ffill().bfill()

In [ ]:
# Ensure chronological order
date_col = 'date' if 'date' in df.columns else 'timestamp'
df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(['city', date_col]).reset_index(drop=True)

# Dynamic City ID Mapping for ALL cities
unique_cities = df['city'].unique()
city_to_id = {city: idx for idx, city in enumerate(unique_cities)}
df['city_id'] = df['city'].map(city_to_id)

print(f"Detected {len(unique_cities)} cities: {list(unique_cities)}")



In [ ]:
# Define Features and Targets
target_cols = ['tmin', 'tmax', 'tavg']
exclude_cols = ['city', date_col] + target_cols
feature_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

# Train Test Split

In [ ]:
train_dfs, test_dfs = [], []
for city in unique_cities:
    city_df = df[df['city'] == city]
    split_idx = int(len(city_df) * 0.8)
    train_dfs.append(city_df.iloc[:split_idx])
    test_dfs.append(city_df.iloc[split_idx:])

train_df = pd.concat(train_dfs).reset_index(drop=True)
test_df = pd.concat(test_dfs).reset_index(drop=True)

#Standarize Data

In [ ]:
cols_to_scale = [c for c in feature_cols if c != 'city_id']

scaler_X = StandardScaler()
scaler_y = StandardScaler()

train_df[cols_to_scale] = scaler_X.fit_transform(train_df[cols_to_scale].astype(float))
test_df[cols_to_scale] = scaler_X.transform(test_df[cols_to_scale].astype(float))

train_df[target_cols] = scaler_y.fit_transform(train_df[target_cols].astype(float))
test_df[target_cols] = scaler_y.transform(test_df[target_cols].astype(float))

# Sequence Window

In [ ]:
def create_city_sequences(data, feature_cols, target_cols, seq_len=30):
    xs, ys, cities_list = [], [], []
    for city in data['city'].unique():
        city_df = data[data['city'] == city]
        X_city = city_df[feature_cols].values
        y_city = city_df[target_cols].values

        for i in range(len(X_city) - seq_len):
            xs.append(X_city[i:(i + seq_len)])
            ys.append(y_city[i + seq_len])
            cities_list.append(city)

    return np.array(xs), np.array(ys), np.array(cities_list)

SEQ_LEN = 30
X_train_seq, y_train_seq, _ = create_city_sequences(train_df, feature_cols, target_cols, seq_len=SEQ_LEN)
X_test_seq, y_test_seq, test_seq_cities = create_city_sequences(test_df, feature_cols, target_cols, seq_len=SEQ_LEN)


# DataSet and DataLoader

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = SequenceDataset(X_train_seq, y_train_seq)
test_dataset = SequenceDataset(X_test_seq, y_test_seq)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model

In [ ]:
class SharedWeatherLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, output_dim=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        last_step_out = out[:, -1, :]
        x = self.dropout(last_step_out)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)

input_dim = X_train_seq.shape[2]
model = SharedWeatherLSTM(input_dim=input_dim, hidden_dim=64, num_layers=1, output_dim=3, dropout=0.2).to(device)

# Train

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 30
patience = 10
best_test_loss = float('inf')
patience_counter = 0
best_model_state = None

print(f"\nTraining Multi-City Shared LSTM model across {len(unique_cities)} cities...")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)

    epoch_train_loss = train_loss / len(train_loader.dataset)

    # Validation evaluation
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            preds = model(batch_X)
            loss = criterion(preds, batch_y)
            test_loss += loss.item() * batch_X.size(0)

    epoch_test_loss = test_loss / len(test_dataset)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {epoch_train_loss:.4f} | Test Loss: {epoch_test_loss:.4f}")

    # Early Stopping Logic
    if epoch_test_loss < best_test_loss:
        best_test_loss = epoch_test_loss
        patience_counter = 0
        best_model_state = model.state_dict()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}! Best Test Loss: {best_test_loss:.4f}")
            model.load_state_dict(best_model_state)
            break

# Eval and Report

In [ ]:
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32).to(device)
    test_preds_scaled = model(X_test_tensor).cpu().numpy()

y_test_actual = scaler_y.inverse_transform(y_test_seq)
test_preds = scaler_y.inverse_transform(test_preds_scaled)

target_names = ['tmin', 'tmax', 'tavg']

print("\n" + "="*85)
print("          MULTI-CITY WEATHER LSTM - PERFORMANCE EVALUATION REPORT")
print("="*85)
print(f"• Total Cities Evaluated : {len(unique_cities)}")
print(f"• Total Test Sequences   : {len(X_test_seq)}")
print("-" * 85)
print(f"{'City':<15} | {'Target':<6} | {'RMSE (°C)':<10} | {'MAE (°C)':<10} | {'R² Score':<10} | {'Accuracy Status':<15}")
print("-" * 85)

overall_actuals = []
overall_preds = []

for city in unique_cities:
    city_mask = (test_seq_cities == city)
    if city_mask.sum() > 0:
        city_actuals = y_test_actual[city_mask]
        city_preds = test_preds[city_mask]

        overall_actuals.append(city_actuals)
        overall_preds.append(city_preds)

        for j, target in enumerate(target_names):
            act = city_actuals[:, j]
            pred = city_preds[:, j]

            rmse = np.sqrt(np.mean((pred - act) ** 2))
            mae = mean_absolute_error(act, pred)
            r2 = r2_score(act, pred)
            acc_status = "Excellent" if r2 > 0.85 else ("Good" if r2 > 0.70 else "Moderate")

            print(f"{city:<15} | {target.upper():<6} | {rmse:<10.4f} | {mae:<10.4f} | {r2:<10.4f} | {acc_status:<15}")
        print("-" * 85)

all_actuals_flat = np.vstack(overall_actuals)
all_preds_flat = np.vstack(overall_preds)

global_rmse = np.sqrt(np.mean((all_preds_flat - all_actuals_flat) ** 2))
global_mae = mean_absolute_error(all_actuals_flat, all_preds_flat)
global_r2 = r2_score(all_actuals_flat, all_preds_flat)

print("\n" + "="*85)
print("                   GLOBAL MULTI-CITY PERFORMANCE SUMMARY")
print("="*85)
print(f"  • Overall Combined RMSE : {global_rmse:.4f} °C")
print(f"  • Overall Combined MAE  : {global_mae:.4f} °C")
print(f"  • Overall Model R² Score: {global_r2:.4f}  ({global_r2 * 100:.2f}% variance explained)")
print("="*85)

# Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt

# Plot predictions vs actuals for each city and target (Main Model Evaluation)
for city in ['Gilgit', 'Quetta']:
    city_mask = (test_seq_cities == city)
    if city_mask.sum() > 0:
        city_actuals = y_test_actual[city_mask]
        city_preds = test_preds[city_mask]

        # Plot a subset (e.g., last 150 days of the test set) for clear visibility
        plot_limit = min(150, len(city_actuals))

        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        fig.suptitle(f'Weather LSTM Predictions vs Actuals - {city} (Last {plot_limit} Days)', fontsize=16, fontweight='bold')

        target_names = ['tmin', 'tmax', 'tavg']
        colors = ['#1f77b4', '#d62728', '#2ca02c'] # Blue for tmin, Red for tmax, Green for tavg

        for j, target in enumerate(target_names):
            ax = axes[j]
            ax.plot(city_actuals[-plot_limit:, j], label=f'Actual {target.upper()}', color=colors[j], linewidth=2, alpha=0.8)
            ax.plot(city_preds[-plot_limit:, j], label=f'Predicted {target.upper()}', color=colors[j], linestyle='--', linewidth=2)

            ax.set_ylabel(f'{target.upper()} (°C)', fontsize=12)
            ax.legend(loc='upper left')
            ax.grid(True, linestyle=':', alpha=0.6)

        plt.xlabel('Test Time Steps (Days)', fontsize=12)
        plt.tight_layout()
        plt.show()

# Save Model

In [ ]:
torch.save(model.state_dict(), 'multi_city_weather_lstm.pth')
joblib.dump(scaler_X, 'scaler_X_multicity.pkl')
joblib.dump(scaler_y, 'scaler_y_multicity.pkl')
joblib.dump(city_to_id, 'city_to_id_mapping.pkl')
print("\n-> Model, scalers, and city mapping saved successfully!")


# Testing

In [ ]:
def predict_next_day_weather_multicity(city_name, recent_30_days_df):
    """
    Takes past 30 days dataframe for ANY supported city, applies scaling,
    and predicts next day tmin, tmax, and tavg using the shared LSTM model.
    """
    model.eval()

    if city_name not in city_to_id:
        raise ValueError(f"City '{city_name}' not found in training mapping. Available: {list(city_to_id.keys())}")

    city_id_val = city_to_id[city_name]
    inf_df = recent_30_days_df.copy()
    inf_df['city_id'] = city_id_val

    features_data = inf_df[feature_cols].copy()
    cols_to_scale_local = [c for c in feature_cols if c != 'city_id']
    features_data[cols_to_scale_local] = scaler_X.transform(features_data[cols_to_scale_local].astype(float))

    X_input = features_data.values.astype(np.float32)
    X_input = np.expand_dims(X_input, axis=0)

    X_tensor = torch.tensor(X_input, dtype=torch.float32).to(device)

    with torch.no_grad():
        pred_scaled = model(X_tensor).cpu().numpy()

    pred_actual = scaler_y.inverse_transform(pred_scaled)

    return {
        'city': city_name,
        'tmin': float(pred_actual[0, 0]),
        'tmax': float(pred_actual[0, 1]),
        'tavg': float(pred_actual[0, 2])
    }

# Testing on Future Data

In [ ]:
print("Loading Saved Model & Scalers for Fine-Tuning...")
print("="*50)

scaler_X = joblib.load('scaler_X_multicity.pkl')
scaler_y = joblib.load('scaler_y_multicity.pkl')
city_to_id = joblib.load('city_to_id_mapping.pkl')

class SharedWeatherLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, output_dim=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_dim)

    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        last_step_out = out[:, -1, :]
        x = self.dropout(last_step_out)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)

# ==========================================
# 2. LOAD & PREPROCESS NEW DATASET
# ==========================================
df_new = pd.read_csv('pakistan_weather_data-Sep2024-Oct2025.csv')

if 'visibility' in df_new.columns:
    df_new = df_new.drop(columns=['visibility'])

df_new = df_new.ffill().bfill()
date_col = 'date' if 'date' in df_new.columns else 'timestamp'
df_new[date_col] = pd.to_datetime(df_new[date_col])
df_new = df_new.sort_values(['city', date_col]).reset_index(drop=True)

df_new = df_new[df_new['city'].isin(city_to_id.keys())].copy()
df_new['city_id'] = df_new['city'].map(city_to_id)

target_cols = ['tmin', 'tmax', 'tavg']
exclude_cols = ['city', date_col] + target_cols
feature_cols = [col for col in df_new.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df_new[col])]

# Train (70%) / Evaluation (30%) Split on New Dataset
ft_train_dfs, ft_eval_dfs = [], []
for city in df_new['city'].unique():
    c_df = df_new[df_new['city'] == city]
    split_idx = int(len(c_df) * 0.7)
    ft_train_dfs.append(c_df.iloc[:split_idx])
    ft_eval_dfs.append(c_df.iloc[split_idx:])

ft_train_df = pd.concat(ft_train_dfs).reset_index(drop=True)
ft_eval_df = pd.concat(ft_eval_dfs).reset_index(drop=True)

# Apply Scalers
cols_to_scale = [c for c in feature_cols if c != 'city_id']
ft_train_df[cols_to_scale] = scaler_X.transform(ft_train_df[cols_to_scale].astype(float))
ft_eval_df[cols_to_scale] = scaler_X.transform(ft_eval_df[cols_to_scale].astype(float))

ft_train_df[target_cols] = scaler_y.transform(ft_train_df[target_cols].astype(float))
ft_eval_df[target_cols] = scaler_y.transform(ft_eval_df[target_cols].astype(float))

# ==========================================
# 3. CREATE SEQUENCES FOR FINE-TUNING
# ==========================================
def create_city_sequences(data, feature_cols, target_cols, seq_len=30):
    xs, ys, cities_list = [], [], []
    for city in data['city'].unique():
        city_df = data[data['city'] == city]
        X_city = city_df[feature_cols].values
        y_city = city_df[target_cols].values

        for i in range(len(X_city) - seq_len):
            xs.append(X_city[i:(i + seq_len)])
            ys.append(y_city[i + seq_len])
            cities_list.append(city)

    return np.array(xs), np.array(ys), np.array(cities_list)

SEQ_LEN = 30
X_ft_train, y_ft_train, _ = create_city_sequences(ft_train_df, feature_cols, target_cols, seq_len=SEQ_LEN)
X_ft_eval, y_ft_eval, eval_seq_cities = create_city_sequences(ft_eval_df, feature_cols, target_cols, seq_len=SEQ_LEN)

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

ft_train_loader = DataLoader(SequenceDataset(X_ft_train, y_ft_train), batch_size=16, shuffle=True)

# ==========================================
# 4. INITIALIZE MODEL & LOAD PREVIOUS WEIGHTS
# ==========================================
input_dim = X_ft_train.shape[2]
model = SharedWeatherLSTM(input_dim=input_dim, hidden_dim=64, num_layers=1, output_dim=3, dropout=0.2).to(device)
model.load_state_dict(torch.load('multi_city_weather_lstm.pth', map_location=device))

# Use smaller learning rate for Fine-Tuning so weights adapt smoothly
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.MSELoss()

# ==========================================
# 5. FINE-TUNING LOOP (10 Epochs)
# ==========================================
print("\nStarting Fine-Tuning on Sep 2024 - Oct 2025 Data...")
epochs_ft = 10

for epoch in range(epochs_ft):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in ft_train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_X.size(0)

    epoch_loss = running_loss / len(ft_train_loader.dataset)
    print(f"Fine-Tuning Epoch [{epoch+1}/{epochs_ft}] | Loss: {epoch_loss:.4f}")

# Save fine-tuned model weights
torch.save(model.state_dict(), 'multi_city_weather_lstm_finetuned.pth')
print("\n-> Fine-tuned model weights saved as 'multi_city_weather_lstm_finetuned.pth'!")

# ==========================================
# 6. EVALUATION OF FINE-TUNED MODEL
# ==========================================
model.eval()
with torch.no_grad():
    X_eval_tensor = torch.tensor(X_ft_eval, dtype=torch.float32).to(device)
    preds_scaled = model(X_eval_tensor).cpu().numpy()

y_actual = scaler_y.inverse_transform(y_ft_eval)
preds_actual = scaler_y.inverse_transform(preds_scaled)

target_names = ['tmin', 'tmax', 'tavg']
unique_test_cities = np.unique(eval_seq_cities)

print("\n" + "="*85)
print("     FINE-TUNED MODEL EVALUATION REPORT (Evaluation Split)")
print("="*85)
print(f"• Total Unseen Sequences Evaluated: {len(X_ft_eval)}")
print("-" * 85)
print(f"{'City':<15} | {'Target':<6} | {'RMSE (°C)':<10} | {'MAE (°C)':<10} | {'R² Score':<10} | {'Accuracy Status':<15}")
print("-" * 85)

overall_actuals = []
overall_preds = []

for city in unique_test_cities:
    city_mask = (eval_seq_cities == city)
    if city_mask.sum() > 0:
        c_actuals = y_actual[city_mask]
        c_preds = preds_actual[city_mask]

        overall_actuals.append(c_actuals)
        overall_preds.append(c_preds)

        for j, target in enumerate(target_names):
            act = c_actuals[:, j]
            pred = c_preds[:, j]

            rmse = np.sqrt(np.mean((pred - act) ** 2))
            mae = mean_absolute_error(act, pred)
            r2 = r2_score(act, pred)
            acc_status = "Excellent" if r2 > 0.85 else ("Good" if r2 > 0.70 else "Moderate")

            print(f"{city:<15} | {target.upper():<6} | {rmse:<10.4f} | {mae:<10.4f} | {r2:<10.4f} | {acc_status:<15}")
        print("-" * 85)

all_act_flat = np.vstack(overall_actuals)
all_pred_flat = np.vstack(overall_preds)

g_rmse = np.sqrt(np.mean((all_pred_flat - all_act_flat) ** 2))
g_mae = mean_absolute_error(all_act_flat, all_pred_flat)
g_r2 = r2_score(all_act_flat, all_pred_flat)

print("\n" + "="*85)
print("             FINE-TUNED GLOBAL PERFORMANCE SUMMARY")
print("="*85)
print(f"  • Overall Combined RMSE : {g_rmse:.4f} °C")
print(f"  • Overall Combined MAE  : {g_mae:.4f} °C")
print(f"  • Overall Model R² Score: {g_r2:.4f}  ({g_r2 * 100:.2f}% variance explained)")
print("="*85)